In [ ]:
from langchain.agents.middleware import PIIMiddleware
pii_middleware_phone=PIIMiddleware(
    "phone_number",
    detector=r"(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{2,4}\)?[\s.-]?)?\d{3,4}[\s.-]?\d{4}",
    strategy="mask"
)
from dotenv import load_dotenv
import os
load_dotenv()
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
model=init_chat_model(
    model="qwen-long",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL")
)
agent =create_agent(
    model=model,
    middleware=[pii_middleware_phone]
)
response=agent.invoke({
    "messages":[HumanMessage("记住我的手机号码15011164761")]
})
for message in response['messages']:
     message.pretty_print()

In [ ]:
from langchain_core.tools import tool
emails = [                                           # 模拟邮箱里的邮件数据
    {
        "subject": "打个招呼",                        # 邮件主题
        "content": """
            你好，张三，
            我们好久没聊天啦，今天突然想起你。
            你最近怎么样？工作和生活一切顺利吗？

            期待你的回信，保重！
        """,                                         # 邮件内容
        "from": "lisi@163.com",                      # 发件人
        "status": "未读"                              # 状态
    },
    {
        "subject": "工作咨询",                        # 邮件主题
        "content": """
            你好，张三，
            最近还好吗？

            不知道你什么时候有空，能不能一起聊聊。不急，有空再回复就好。

            祝你一切顺心！
        """,                                         # 邮件内容
        "from": "wangwu@163.com",                    # 发件人
        "status": "已读"                              # 状态
    }
]
@tool
def check_email()->str:
    """查看邮件"""
    return str(emails)
@tool
def send_email(to:str,subject:str,content:str):
    """回复邮件"""
    return f"邮件已发送给{to},主题：{subject},内容：{content}"
from langchain.agents.middleware import HumanInTheLoopMiddleware #导入人工审核中间件
email_middleware=HumanInTheLoopMiddleware(
    interrupt_on={                          #指出哪些工具调用时终断等待人工
        "send_email":{
            "description":"请确认邮件内容",
            "allowed_decisions":["approve","reject"]
        }
    }
)
from dotenv import load_dotenv
import os
load_dotenv()
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import  InMemorySaver
from langchain_core.messages import HumanMessage
model=init_chat_model(
    model="qwen3.5-plus",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL")
)
agent=create_agent(
    model,
    tools=[check_email,send_email],
    middleware=[email_middleware],
    checkpointer=InMemorySaver(),
    system_prompt="我是一个邮件助手"
)
config={"configurable":{"thread_id":"1"}}
response=agent.invoke(
    {"messages":[HumanMessage("帮我查看邮件")]},
    config
)
for message in response["messages"]:
     message.pretty_print()
response=agent.invoke({
    "messages":[HumanMessage("帮我回复李四，问他具体时间")]
   },
    config
)
for message in response["messages"]:
     message.pretty_print()

In [ ]:
from langgraph.types import Command
response=agent.invoke(
    Command(
        resume={
           "decisions":[
               {
                   "type": "reject",
                   "message": "语气太正式了，我和李四是好朋友，用更亲切的语气回复"
               }
           ]
        }
    ),
    config
)
for message in response["messages"]:
    message.pretty_print()
response=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {
                        "type": "approve",
                    }
                ]
            }
        ),config
    )
for message in response["messages"]:
    message.pretty_print()

In [ ]:
response=agent.invoke({
    "messages":[HumanMessage("把给李四的消息发给小杨一下 邮箱地址是1152718127@qq.com")]
   },
    config
)
for message in response["messages"]:
    message.pretty_print()
response=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {
                       "type": "reject",
                       "message": "发给小杨邮件字数控制在150子左右"
                   }
                ]
            }
        ),config
    )
for message in response["messages"]:
    message.pretty_print()
response=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {
                        "type": "approve",
                    }
                ]
            }
        ),config
    )
for message in response["messages"]:
    message.pretty_print()

In [ ]:

response=agent.invoke({
    "messages":[HumanMessage("邮箱地址是1152718127@qq.com的用户真实的邮箱为啥没有收到邮件呢 这个是腾讯qq邮箱地址")]
   },
    config
)

for message in response['messages']:
    message.pretty_print()
